# RAG Fusion

An implementation of **RAG-Fusion**, a retrieval-augmented generation technique that improves
on standard RAG by querying the vector store with *multiple* reformulations of the user's
question and merging the results with **Reciprocal Rank Fusion (RRF)** before generating an
answer.

**Why it helps:** a single query embedding can miss relevant chunks phrased differently than
the question. Generating several related queries (via an LLM) and fusing their retrieved
results surfaces documents that multiple query variants agree are relevant, producing a more
robust context for the final answer.

**Pipeline:**
1. **Load** a source document from the web.
2. **Chunk** it into retrievable passages.
3. **Embed** the chunks into a vector store (Chroma + a sentence-transformers model).
4. **Generate multiple queries** from the user's question with a local LLM (Ollama).
5. **Retrieve & fuse** results across all queries with RRF.
6. **Generate** a final answer grounded in the fused context.

**Stack:** LangChain, Chroma, HuggingFace `sentence-transformers/all-MiniLM-L6-v2` embeddings,
and Ollama (`phi3`) as the local LLM.

> Source document used in this example: [*LLM Powered Autonomous Agents*](https://lilianweng.github.io/posts/2023-06-23-agent/) by Lilian Weng.


## 1. Load Documents

Fetch the source article and strip it down to just the post content, title, and header.

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

## 2. Chunk the Document

Split the loaded document into overlapping ~500-token chunks so that each chunk is small enough to embed and retrieve individually, while the overlap preserves context across chunk boundaries.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_overlap=30, chunk_size=500)
chunks = splitter.split_documents(blog_docs)
print("No. of Chunks: ",len(chunks))
chunks[0]

## 3. Embed & Index

Embed each chunk with a sentence-transformers model and store the vectors in a local Chroma vector store. `retriever` will later fetch the single most relevant chunk (`k=1`) per query.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(embedding=embed,documents=chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

## 4. Generate Multiple Queries

This is the core idea of RAG-Fusion: instead of retrieving with just the user's original question, ask an LLM to generate several related search queries. Each will be run against the retriever independently in the next step.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

# RAG-Fusion: Related
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

llm = ChatOllama(model="phi3",temperature=0)

generate_queries = prompt_rag_fusion | llm | StrOutputParser() | (lambda x: x.split("\n"))

## 5. Reciprocal Rank Fusion

Each generated query returns its own ranked list of retrieved documents. RRF merges these lists into one ranking: a document scores `1 / (rank + k)` in each list it appears in, and scores are summed across lists. Documents that rank well across *multiple* queries end up ranked highest overall -- which is more robust than trusting any single query's retrieval.

In [ ]:
from langchain_core.load import loads, dumps


def reciprocal_rank_fusion(documents: list[list], k: int = 60):
    """Fuse multiple ranked lists of documents into a single ranking.

    Reciprocal Rank Fusion (RRF) combines several ranked result lists
    (e.g. retrieved documents for several reformulated queries) into one
    list, boosting documents that consistently appear near the top across
    multiple lists rather than just documents that top a single list.

    For every occurrence of a document at position `rank` (0-indexed) in
    one of the input lists, it receives a score of 1 / (rank + k). Scores
    for the same document are summed across all lists.

    Args:
        documents: A list of ranked lists of documents (e.g. one list per
            generated query, each already ordered by retriever relevance).
        k: Smoothing constant from the RRF formula. Higher k reduces the
            influence of any single top rank, so the fused order depends
            more on cross-list agreement than any one list's extremes.

    Returns:
        A list of (document, score) tuples sorted by descending fused
        score -- the RAG-Fusion re-ranking of all input documents.
    """
    fused_scores: dict[str, float] = {}

    for docs in documents:
        for rank, doc in enumerate(docs):
            # Serialize so identical documents from different query runs
            # are recognized as the same key and their scores accumulate.
            doc_str = dumps(doc)
            fused_scores.setdefault(doc_str, 0)
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results


# Full RAG-Fusion retrieval pipeline: generate several queries, retrieve
# for each, then fuse the results into one ranked context.
fusion_chain = generate_queries | retriever.map() | reciprocal_rank_fusion


## 6. Retrieval-Augmented Generation

Finally, wire the fused retrieval results into a prompt template and ask the LLM to answer the original question using only that fused context.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# RAG
template = """
    Answer the following question based on this context:
    {context}

    Question:
    {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": fusion_chain, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

question = "What is task decomposition in LLMs?"
final_rag_chain.invoke({"question":question})

'Task decomposition in LLMs refers to the process of breaking down a complex task into smaller, more manageable steps that can be easily understood and executed by the language model. This technique is often used to enhance the performance of LLMs on complex tasks. There are several ways to achieve task decomposition in LLMs:\n\n1. Simple prompting: The LLM is instructed to provide step-by-step instructions for completing a task. For example, the LLM might be prompted to provide a list of steps for planning a trip or writing a story outline.\n\n2. Task-specific instructions: The LLM is given specific instructions for completing a task that is common in a particular domain. For example, the LLM might be given instructions for writing a novel or creating a scientific research paper.\n\n3. Human inputs: The LLM can also receive inputs from a human user, who can provide guidance and feedback on the task decomposition process. This approach can be particularly useful for tasks that require 

## Next Steps

- Swap in a different retriever (e.g. hybrid BM25 + vector search) to see RRF fuse across genuinely different retrieval strategies, not just query paraphrases.
- Try a larger/remote LLM in place of local `phi3` for stronger query generation and final answers.
- Add evaluation (e.g. retrieval precision/recall or answer faithfulness) to quantify RAG-Fusion's improvement over single-query RAG on a held-out question set.